<a href="https://colab.research.google.com/github/jayanthbagare/build_a_large_language_model/blob/main/llm_ch02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
! pip install tiktoken

In [4]:
import re

In [5]:
try:
    with open("the-verdict.txt","r",encoding="utf-8") as f:
        raw_text = f.read()
except FileNotFoundError:
    ! wget https://raw.githubusercontent.com/jayanthbagare/build_a_large_language_model/refs/heads/main/the-verdict.txt
    with open("the-verdict.txt","r",encoding="utf-8") as f:
        raw_text = f.read()

--2026-03-28 15:27:47--  https://raw.githubusercontent.com/jayanthbagare/build_a_large_language_model/refs/heads/main/the-verdict.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 20481 (20K) [text/plain]
Saving to: ‘the-verdict.txt’

the-verdict.txt     100%[===================>]  20.00K  --.-KB/s    in 0s      

2026-03-28 15:27:47 (117 MB/s) - ‘the-verdict.txt’ saved [20481/20481]



In [6]:
print("Total Characters:",len(raw_text))
print(raw_text[:99])
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)',raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))

Total Characters: 20481
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 
4690


In [7]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [8]:
vocab = {token:integer for integer,token in enumerate(all_words)}
for i,item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [9]:
class SimpleTokenizerV1:
    def __init__(self,vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self,text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)',text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self,ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])',r'\1',text)
        return text

In [10]:
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know, Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)

In [11]:
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 67, 7, 38, 851, 1108, 754, 793, 7]


In [12]:
print(tokenizer.decode(ids))

" It' s the last he painted, you know, Mrs. Gisburn said with pardonable pride.


In [13]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>","<|unk|>"])
vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [14]:
print(len(vocab.items()))

1132


In [15]:
class SimpleTokenizerV2:
    def __init__(self,vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self,text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)',text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self,ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])',r'\1',text)
        return text

In [16]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = "<|endoftext|> ".join((text1,text2))
print(text)

Hello, do you like tea?<|endoftext|> In the sunlit terraces of the palace.


In [17]:
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [18]:
print(tokenizer.decode(tokenizer.encode(text)))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


In [19]:
from importlib.metadata import version
import tiktoken
print("tiktoken.version:", version("tiktoken"))

tiktoken.version: 0.12.0


In [20]:
#Tiktoken tokenizer
tokenizer = tiktoken.get_encoding("gpt2")
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of someunknowPlace."
text = "<|endoftext|> ".join((text1,text2))
print(text)
integers = tokenizer.encode(text,allowed_special={"<|endoftext|>"})
print(integers)
string = tokenizer.decode(integers)
print(string)

Hello, do you like tea?<|endoftext|> In the sunlit terraces of someunknowPlace.
[15496, 11, 466, 345, 588, 8887, 30, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 2954, 2197, 27271, 13]
Hello, do you like tea?<|endoftext|> In the sunlit terraces of someunknowPlace.


In [21]:
text = "VenyaBagare"
e = tokenizer.encode(text)
print(e)
for i in e:
  print(tokenizer.decode([i]))

[37522, 3972, 33, 363, 533]
Ven
ya
B
ag
are


In [22]:
#Sliding Window Data Sampling
with open("the-verdict.txt","r",encoding="utf-8") as f:
  raw_text = f.read()
tokenizer = tiktoken.get_encoding("gpt2")
enc_text = tokenizer.encode(raw_text)

print(len(raw_text))
enc_sample = enc_text[50:]

20481


In [23]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:  {y}")

x: [290, 4920, 2241, 287]
y:  [4920, 2241, 287, 257]


In [24]:
for i in range(1,context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]
  print(context,"---->",desired)
  print(tokenizer.decode(context),"------>",tokenizer.decode([desired]))

[290] ----> 4920
 and ------>  established
[290, 4920] ----> 2241
 and established ------>  himself
[290, 4920, 2241] ----> 287
 and established himself ------>  in
[290, 4920, 2241, 287] ----> 257
 and established himself in ------>  a


In [25]:
#Data Loading as tensors via torch
import torch
from torch.utils.data import Dataset,DataLoader

In [26]:
class GPTDatasetV1(Dataset):
  def __init__(self,txt,tokenizer,max_length,stride):
    self.input_ids = []
    self.target_ids = []

    token_ids = tokenizer.encode(txt)

    for i in range(0,len(token_ids) - max_length,stride):
      input_chunk = token_ids[i:i + max_length]
      target_chunk = token_ids[i + 1 : i + max_length + 1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))
  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self,idx):
    return self.input_ids[idx],self.target_ids[idx]

In [27]:
def create_dataloader_v1(txt,batch_size=4,max_length=256,stride=128,shuffle=True,drop_last=True,num_workers=0):
  tokenizer = tiktoken.get_encoding("gpt2")
  dataset = GPTDatasetV1(txt,tokenizer,max_length,stride)
  dataloader = DataLoader(dataset,batch_size=batch_size,shuffle=shuffle,drop_last=drop_last,num_workers=num_workers)
  return dataloader

In [28]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
  raw_text = f.read()
dataloader = create_dataloader_v1(raw_text,batch_size=8,max_length=4,stride=4,shuffle=False)
data_iter = iter(dataloader)
inputs,targets = next(data_iter)
print("Inputs:\n",inputs)
print("Targets:\n",targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


In [45]:
vocab_size = tokenizer.n_vocab
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size,output_dim)

In [46]:
max_length = 4
batch_size = 8
dataloader = create_dataloader_v1(raw_text,batch_size=batch_size,max_length=max_length,stride=max_length,shuffle=False)
data_iter = iter(dataloader)
input,targets = next(data_iter)
print("\nInput shape:\n",input.shape)


Input shape:
 torch.Size([8, 4])
